# EnzyExtract

This tutorial describes EnzyExtract, a pipeline to extract $k_{cat}$ and $K_M$ values using LLMs.

It accompanies the manuscript, *Finding the Dark Matter: Large Language Model-based Enzyme Kinetic Data Extractor and Its Validation*. A preprint is available on [chemrxiv](https://chemrxiv.org/engage/chemrxiv/article-details/6801df4850018ac7c5f340a1).

## Step ii: Download papers

Place your papers in /content/papers. Alternatively, use the example paper.

The example paper is *Arsinothricin, an arsenic-containing
 non-proteinogenic amino acid analog of glutamate,
 is a broad-spectrum antibiotic*, by Nadar et al., which is available under [CC BY 4.0](http://creativecommons.org/licenses/by/4.0/).


 Papers should be given a unique filename, and we recommend using the PubMed ID.


In [ ]:
!mkdir /content/papers
!wget -O /content/papers/30993215.pdf https://www.nature.com/articles/s42003-019-0365-y.pdf

--2025-06-13 18:25:30--  https://www.nature.com/articles/s42003-019-0365-y.pdf
Resolving www.nature.com (www.nature.com)... 151.101.0.95, 151.101.64.95, 151.101.128.95, ...
Connecting to www.nature.com (www.nature.com)|151.101.0.95|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://idp.nature.com/authorize?response_type=cookie&client_id=grover&redirect_uri=https%3A%2F%2Fwww.nature.com%2Farticles%2Fs42003-019-0365-y.pdf [following]
--2025-06-13 18:25:31--  https://idp.nature.com/authorize?response_type=cookie&client_id=grover&redirect_uri=https%3A%2F%2Fwww.nature.com%2Farticles%2Fs42003-019-0365-y.pdf
Resolving idp.nature.com (idp.nature.com)... 151.101.0.95, 151.101.64.95, 151.101.128.95, ...
Connecting to idp.nature.com (idp.nature.com)|151.101.0.95|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://idp.nature.com/transit?redirect_uri=https%3A%2F%2Fwww.nature.com%2Farticles%2Fs42003-019-0365-y.pdf&code=60b96e8

## Step i: Installation





In [ ]:
!pip install git+https://github.com/conjuncts/gmft_pymupdf -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 56.8 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/ChemBioHTP/EnzyExtract
%cd EnzyExtract
!git pull origin main
!pip install -e .
%cd /content/

fatal: destination path 'EnzyExtract' already exists and is not an empty directory.
/content/EnzyExtract
From https://github.com/ChemBioHTP/EnzyExtract
 * branch            main       -> FETCH_HEAD
Already up to date.
Obtaining file:///content/EnzyExtract
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for enzyextract (pyproject.toml) ... done
  Created wheel for enzyextract: filename=enzyextract-0.0.1-py3-none-any.whl size=2285 sha256=d1686dc1a61cf55f0675f9d028a2618c623482fe16aa95aa67caeecf113d3691
  Stored in directory: /tmp/pip-ephem-wheel-cache-dleut344/wheels/b2/d5/11/184b62764eb79b650ca5a65b938d1690b8e323467ed4acbb52
Successfully built enzyextract
  Attempting uninstall: enzyextract
    Found existing installation: enzyextract 0.0.1
    Uninstalling enzyextract-0.0.1:
      Successfully uninstalled enzy

In [ ]:
!pwd

/content


In [ ]:
!pip show enzyextract

Name: enzyextract
Version: 0.0.1
Summary: EnzyExtract successor
Home-page: 
Author: conjuncts
Author-email: 
License: 
Location: /usr/local/lib/python3.11/dist-packages
Editable project location: /content/EnzyExtract
Requires: anthropic, biopython, gmft, google-cloud-storage, litellm, pandas, polars, ryaml, seaborn, tenacity
Required-by: 


Note: you may need to follow [these](
https://stackoverflow.com/questions/57838013/modulenotfounderror-after-successful-pip-install-in-google-colaboratory) special steps for the editable install.


In [ ]:
import site
site.main()
import importlib
import enzyextract

In [ ]:
# reload modules
import importlib
import sys
def refresh_modules():
    _modules = sys.modules.copy()
    for module in _modules.values():
        if 'enzyextract' in str(module):
            importlib.reload(module)
refresh_modules()

## Step 0: Preprocessing

Code can be found in `experiments/example/pipeline/ex_step0_run_preprocessing.py`.

Click the "eye" in the left panel to see the hidden `.enzy` folder. [See here](https://stackoverflow.com/questions/67698933/how-to-show-hidden-files-colab).

In [ ]:
import os
from enzyextract.pre.reocr.m_mu_reocr import script_scan_mM
from enzyextract.pre.scans.scan_to_parquet import scan_papers
from enzyextract.pre.table.scan_tables import process_pdfs

if __name__ == '__main__':

    pdf_root = '/content/papers' # PDFs to process
    enzy_root = '/content/.enzy' # where intermediate data for these PDFs is stored

    print("Starting mM...")
    script_scan_mM(
        pdf_root=pdf_root,
        write_dir=f'{enzy_root}/pre/mM',
        model_path='EnzyExtract/data/models/resnet18-remicro-iter3.pth',
    )

    print("Starting tables...")
    process_pdfs(
        pdf_root=pdf_root,
        write_dir=f"{enzy_root}/pre/tables",
        micros_path=f"{enzy_root}/pre/mM/mM.parquet",
        # _check_nonzero_tables=False,
    )

    print(f"Compressing PDFs to {enzy_root}/scans/pdf/pdf.parquet")
    df = scan_papers(
        pdfs_folder=pdf_root,
        recursive=False,
    )
    os.makedirs(f'{enzy_root}/scans/pdf', exist_ok=True)
    df.write_parquet(f'{enzy_root}/scans/pdf/pdf.parquet')

Starting mM...


100%|██████████| 1/1 [00:03<00:00,  3.37s/it]


Starting tables...
Making directory /content/.enzy/pre/tables
Adding suffix to correction_df
Common pdfs: 0 / 1


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/76.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

100%|██████████| 1/1 [00:17<00:00, 17.48s/it]


Compressing PDFs to /content/.enzy/scans/pdf/pdf.parquet


100%|██████████| 1/1 [00:00<00:00,  6.71it/s]


In [ ]:
!ls /content/papers

30993215.pdf


In [ ]:
from IPython.display import display, Markdown

with open("/content/.enzy/pre/tables/markdown/30993215_0.md") as f:
  display(Markdown(f.read()))

Table 1 PpArsN1 is selective for L-AST over other glutamine synthetase inhibitors

| Substrate   | (50 μM)   | Speciﬁc activity   | (nmol s−1   | mg−1 PpArsN1)     |
|:------------|:----------|:-------------------|:------------|:------------------|
| AST         |           | 49.6 ± 0.8         |             |                   |
| L-PPT       |           | 13.9 ± 1.9         |             |                   |
| L-MSO       |           | 2.1 ± 0.1          |             |                   |
| Enzyme      | Substrate | Km (μM)            | Kcat (s−1)  | Kcat/Km (M−1 s−1) |
| PpArsN1     | AST       | 11 ± 3             | 1.7 ± 0.2   | 1.55 × 105        |
|             | L-PPT     | 1000 ± 200         | 9.6 ± 0.9   | 0.10 × 105        |
| SvPAT       | AST       | 12 ± 2             | 2.3 ± 0.1   | 1.92 × 105        |
|             | L-PPT     | 47 ± 2             | 3.1 ± 0.0   | 0.66 × 105        |



## Step 1: Submission

You will need to set your `OPENAI_API_KEY` in Google Colab's Secrets panel. You may need to [sign up](https://platform.openai.com/api-keys) with OpenAI. You can also call `process_env('.env')`.

By default, EnzyExtract uses the Batch API.

The batch API is the best method for large volumes of papers, since OpenAI and other vendors offer a 50% discount and higher volumes can be processed than synchronously.

However, the Batch API does not work well with Google Colab. The colab session may time out before the batch completes, in which case the `.enzy` metadata and correspondences between custom_ids and PMIDs (`.enzy/corresp`) will be lost.

When submitting the file, you will have a couple of options.
- `l` (**local**) **is recommended**. This will save a local copy. Then, see the below instructions to run synchronously.
- `y`(**yes**) will use the Batch API. This is recommended for large scale processing, but not for Google Colab.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
from enzyextract.pipeline.step1_run_tableboth import process_env, step1_main
from enzyextract.utils.namespace_management import glean_model_name
from enzyextract.utils import prompt_collections

if __name__ == '__main__':

    llm_provider = 'openai'
    model_name = 'gpt-4o-2024-08-06'
    suggested_prompt = prompt_collections.table_oneshot_v3
    structured = False

    namespace = 'my-namespace-here' # no colons: needs to be a valid file name
    pdf_root = '/content/papers'
    enzy_root = '/content/.enzy'
    step1_main(
        namespace=namespace,
        pdf_root=pdf_root,
        micro_path=f'{enzy_root}/pre/mM/mM.parquet',
        tables_from=f'{enzy_root}/pre/tables/markdown',

        dest_folder=f'{enzy_root}/batches',
        corresp_folder=f'{enzy_root}/corresp',
        log_location=f'{enzy_root}/llm_log.tsv',
        model_name=model_name,
        llm_provider=llm_provider,
        prompt=suggested_prompt,
        structured=structured,

        # _check_nonzero_tables=False,
        _check_nonzero_reocr=False,
    )

Namespace:  my-namespace-here
Intersection of 1 pmids with tables
Adding suffix to correction_df
Common pdfs: 0 / 1
Intersection of 0 pmids with micro corrections


100%|██████████| 1/1 [00:00<00:00, 10.97it/s]


Found 1 pmids with tables
Using model gpt-4o-2024-08-06
Time to submit!
Batch of 1 items at /content/.enzy/batches/my-namespace-here_v1.jsonl ready for submission. Submit to openai?
Proceed? ([y]es, [l]ocal, [u]ntrack, [r]emove, [h]elp): l
Tracked local copy at /content/.enzy/batches/my-namespace-here_v1.jsonl


Check your batches like so.

In [ ]:
from enzyextract.pipeline.llm_log import read_log
read_log('/content/.enzy/llm_log.tsv')

namespace,version,shard,status,model_name,llm_provider,structured,prompt_hash,file_uuid,batch_uuid,batch_fpath,corresp_fpath,completion_fpath
str,str,u32,str,str,str,bool,str,str,str,str,str,str
"""my-namespace-here""","""v1""",null,"""local""","""gpt-4o-2024-08-06""","""openai""",false,"""CnAusrjBG36lihf1tFVRoIGU4yDwBU…",null,null,"""/content/.enzy/batches/my-name…","""/content/.enzy/corresp/my-name…",null


## Step 2: Download

If you are in Google Colab and are processing synchronously, see below.

In [ ]:
from enzyextract.submit.openai_synch import process_batch_synchronously

process_batch_synchronously(
    batch_fpath='/content/.enzy/batches/my-namespace-here_v1.jsonl',
    enzy_root='/content/.enzy'
)

Wrote to /content/.enzy/completions/my-namespace-here_v1.jsonl
Updated /content/.enzy/llm_log.tsv


'/content/.enzy/completions/my-namespace-here_v1.jsonl'

In [ ]:
from enzyextract.pipeline.llm_log import read_log
read_log('/content/.enzy/llm_log.tsv')

namespace,version,shard,status,model_name,llm_provider,structured,prompt_hash,file_uuid,batch_uuid,batch_fpath,corresp_fpath,completion_fpath
str,str,u32,str,str,str,bool,str,str,str,str,str,str
"""my-namespace-here""","""v1""",null,"""downloaded""","""gpt-4o-2024-08-06""","""openai""",false,"""CnAusrjBG36lihf1tFVRoIGU4yDwBU…",null,null,"""/content/.enzy/batches/my-name…","""/content/.enzy/corresp/my-name…","""/content/.enzy/completions/my-…"


### Option 2b (Batch API)

If you used the Batch API, see below.

In [ ]:
from enzyextract.pipeline.step2_download import process_env, download

if __name__ == "__main__":
    process_env('.env')
    download(
        log_location="/content/.enzy/llm_log.tsv",
        dest_folder="/content/.enzy/completions",
        err_folder="/content/.enzy/errors",
    )

In [ ]:
!zip -r enzy.zip /content/.enzy

updating: content/.enzy/ (stored 0%)
  adding: content/.enzy/post/ (stored 0%)
  adding: content/.enzy/post/valid/ (stored 0%)
  adding: content/.enzy/corresp/ (stored 0%)
  adding: content/.enzy/corresp/my-namespace-here_v1.parquet (deflated 58%)
  adding: content/.enzy/scans/ (stored 0%)
  adding: content/.enzy/scans/pdf/ (stored 0%)
  adding: content/.enzy/scans/pdf/pdf.parquet (deflated 61%)
  adding: content/.enzy/batches/ (stored 0%)
  adding: content/.enzy/batches/my-namespace-here_v1.jsonl (deflated 64%)
  adding: content/.enzy/pre/ (stored 0%)
  adding: content/.enzy/pre/tables/ (stored 0%)
  adding: content/.enzy/pre/tables/markdown/ (stored 0%)
  adding: content/.enzy/pre/tables/markdown/30993215_0.md (deflated 62%)
  adding: content/.enzy/pre/tables/seen.txt (stored 0%)
  adding: content/.enzy/pre/tables/info/ (stored 0%)
  adding: content/.enzy/pre/tables/info/30993215_0.info (deflated 55%)
  adding: content/.enzy/pre/tables/false_positives/ (stored 0%)
  adding: content/.

## Step 3: Convert to DataFrame

We use:
- polars for faster performance
- parquet for smaller file sizes, null safety, and nested column types

In [ ]:
import polars as pl
import os

from enzyextract.dependency.prereqs import export
from enzyextract.pipeline.step3_llm_to_df import namespace_to_parquet

if __name__ == '__main__':
    enzy_root = '/content/.enzy'
    df = namespace_to_parquet(
        namespace='my-namespace-here',
        log_location=f'{enzy_root}/llm_log.tsv',
        write_dir=f'{enzy_root}/post/valid'
    )

Using namespace: my-namespace-here version: v1
Strange kcat units set()
Strange km units set()
Writing to /content/.enzy/post/valid/my-namespace-here_v1.parquet


In [ ]:
df

,pmid,enzyme,enzyme_full,substrate,substrate_full,mutant,organism,kcat,km,kcat_km,temperature,pH,solution,cofactors,other,descriptor,custom_id,flag.regurgitation
0,30993215,PpArsN1,PpArsN1,AST,Arsinothricin,None,Pseudomonas putida KT2440,1.7 ± 0.2 s^-1,11 ± 3 µM,1.55 × 10^5 M^-1 s^-1,None,None,None,None,None,PpArsN1,my-namespace-here_v1_30993215,None
1,30993215,PpArsN1,PpArsN1,L-PPT,L-Phosphinothricin,None,Pseudomonas putida KT2440,9.6 ± 0.9 s^-1,1000 ± 200 µM,0.10 × 10^5 M^-1 s^-1,None,None,None,None,None,PpArsN1,my-namespace-here_v1_30993215,None
2,30993215,SvPAT,Streptomyces viridochromogenes phosphinothrici...,AST,Arsinothricin,None,Pseudomonas putida KT2440,2.3 ± 0.1 s^-1,12 ± 2 µM,1.92 × 10^5 M^-1 s^-1,None,None,None,None,None,SvPAT,my-namespace-here_v1_30993215,None
3,30993215,SvPAT,Streptomyces viridochromogenes phosphinothrici...,L-PPT,L-Phosphinothricin,None,Pseudomonas putida KT2440,3.1 ± 0.0 s^-1,47 ± 2 µM,0.66 × 10^5 M^-1 s^-1,None,None,None,None,None,SvPAT,my-namespace-here_v1_30993215,None
